# 02 — Generate 3-Arm Grounding Artifacts (Stage 2 → Stage 3)

Untuk tiap lesion (deteksi GT DENTEX): SAM+adapter → mask, lalu render **bbox / mask / hybrid**. Output ke Drive `outputs/artifacts/`.

**Prasyarat:** `adapter_best.pth` ada di Drive (hasil `01_sam_adapter_train`).

Pakai EVAL SET stratified (n per kelas) supaya biaya GPT-4o Stage 3 terkendali. Runtime: **GPU** (encode ViT-H per gambar).

## Cell 1 — Mount Drive + sync repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/opg-live'

import os
REPO = '/content/opg-live'
if not os.path.exists(REPO):
    !git clone https://github.com/AndreasTopuh/opg-live.git {REPO}
!cd {REPO} && git fetch origin && git reset --hard origin/main && git log --oneline -1

## Cell 2 — Install deps

In [ ]:
%pip install -q segment-anything pycocotools opencv-python-headless
import torch; print('GPU:', torch.cuda.get_device_name(0))

## Cell 3 — Generate artifacts
`--n_per_class 40` → 160 lesion × 3 arm. Naik/turunkan sesuai budget GPT-4o.

In [ ]:
!python /content/opg-live/scripts/make_artifacts.py \
  --drive {DRIVE_ROOT} \
  --sam_ckpt {DRIVE_ROOT}/checkpoints/sam_vit_h_4b8939.pth \
  --adapter {DRIVE_ROOT}/checkpoints/adapter_best.pth \
  --n_per_class 40

## Cell 4 — Preview: 1 lesion, 3 arm berdampingan (sanity check visual)

In [ ]:
import json, matplotlib.pyplot as plt, cv2
ART = f'{DRIVE_ROOT}/outputs/artifacts'
man = [json.loads(l) for l in open(f'{ART}/manifest.jsonl')]
print('Total artifacts:', len(man))

m = man[0]
fig, ax = plt.subplots(1, 3, figsize=(20, 6))
for i, arm in enumerate(['bbox', 'mask', 'hybrid']):
    im = cv2.cvtColor(cv2.imread(f"{ART}/{m['artifacts'][arm]}"), cv2.COLOR_BGR2RGB)
    ax[i].imshow(im); ax[i].set_title(f"{arm} — {m['cls_name']}"); ax[i].axis('off')
plt.tight_layout(); plt.show()

## ✅ Deliverable
- [ ] `outputs/artifacts/{bbox,mask,hybrid}/*.png` di Drive
- [ ] `manifest.jsonl` (lesion_id, kelas, bbox, area, path 3 arm)
- [ ] preview visual 3 arm terlihat benar (box hijau, mask merah, hybrid keduanya)

**Next:** `03_explain_rag.ipynb` (Stage 3) — kirim tiap artifact ke GPT-4o (via OpenRouter) + RAG, hasilkan penjelasan L-F-V, lalu hitung HR/GS/CTC per arm.